# 04 · Ground answers on your own documents (File Search)
**Time:** 20 min · **Goal:** answer from a document and show where the answer came from.

Run the cells from top to bottom. **Do not share your key and do not upload private data.** Free-tier content may be used by Google to improve its products.

Get a key: https://aistudio.google.com/apikey

> File Search is a managed RAG tool in the Gemini API. Syntax can change, so check https://ai.google.dev/gemini-api/docs if a cell fails.

In [ ]:
# Setup: install the Gemini SDK and load your free AI Studio key
!pip -q install google-genai
import os, getpass
try:
    from google.colab import userdata          # Colab: left menu > Secrets > add GOOGLE_API_KEY
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass.getpass("Paste your AI Studio key: ")
from google import genai
from google.genai import types
client = genai.Client(api_key=API_KEY)
# Model names retire quickly. This tries several free-tier models and uses the first that works.
CANDIDATE_MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.5-flash"]
def pick_model():
    for m in CANDIDATE_MODELS:
        try:
            client.models.generate_content(model=m, contents="ping")
            print("Using model:", m)
            return m
        except Exception as e:
            print("Skipping", m, "->", str(e)[:100])
    raise RuntimeError("No model worked. Check your key, account and the model list at https://ai.google.dev/gemini-api/docs/models")
MODEL = pick_model()

## Step 1 · A small sample guide
This is **sample text for practice, not official advice.** Replace it with a real public leaflet from your agriculture ministry or extension service.

In [ ]:
guide = """Maize care guide (practice sample)
Yellow leaves: can be caused by too little nitrogen, waterlogged soil, or pests. Check drainage first, then ask an extension officer about fertiliser.
Fall armyworm: small holes and ragged leaves, droppings in the whorl. Inspect plants early in the morning and report outbreaks to the extension officer.
Majani ya manjano: yanaweza kusababishwa na upungufu wa naitrojeni, maji kutuama au wadudu. Angalia mifereji ya maji kwanza.
"""
open("maize_guide.txt", "w").write(guide)

## Step 2 · Create a store and upload the file

In [ ]:
import time
store = client.file_search_stores.create(config={"display_name": "shamba-guide"})
op = client.file_search_stores.upload_to_file_search_store(
    file="maize_guide.txt", file_search_store_name=store.name, config={"display_name": "maize guide"})
while not op.done:
    time.sleep(3)
    op = client.operations.get(op)
print("Ready:", store.name)

## Step 3 · Ask a question that only the document can answer

In [ ]:
r = client.models.generate_content(
    model=MODEL,
    contents="Majani ya mahindi yanageuka manjano. Sababu gani zinawezekana?",
    config=types.GenerateContentConfig(
        tools=[types.Tool(file_search=types.FileSearch(file_search_store_names=[store.name]))]),
)
print(r.text)
print("\nSources:", r.candidates[0].grounding_metadata)

## Step 4 · Clean up
Delete the store when you finish.

In [ ]:
client.file_search_stores.delete(name=store.name, config={"force": True})

## Your turn
Upload a real public leaflet (no private data). Ask three questions the leaflet answers and one it does not. Does the agent say it does not know?